# MDR_own.ipynb

Original Bi-CoT research notebook, recovered from the author’s AI_Study archive. Outputs, execution state, and machine metadata were removed; API credentials now come from `OPENAI_API_KEY`, and local paths now use `../data/workspace` relative to this notebook directory. See `docs/WORKFLOW.md` before selecting cells. Cells include experimental alternatives and data writes; this is not a single Run All pipeline.


### Historical setup / cleanup cell (archived, not executed)

The original environment-specific cell is preserved below. Configure the documented environment and workspace instead of running this cell.

```python
pip install --upgrade openai
```


### Historical setup / cleanup cell (archived, not executed)

The original environment-specific cell is preserved below. Configure the documented environment and workspace instead of running this cell.

```python
from google.colab import drive
drive.mount('../data/workspace')
```


### Historical setup / cleanup cell (archived, not executed)

The original environment-specific cell is preserved below. Configure the documented environment and workspace instead of running this cell.

```python
!pip install transformers==2.11.0 \
            tensorboard>=1.15.0 \
            numpy \
            tqdm \
            ujson \
            streamlit
```


### Historical setup / cleanup cell (archived, not executed)

The original environment-specific cell is preserved below. Configure the documented environment and workspace instead of running this cell.

```python
cd ../data/workspace/multihop_dense_retrieval
```


### Historical setup / cleanup cell (archived, not executed)

The original environment-specific cell is preserved below. Configure the documented environment and workspace instead of running this cell.

```python
!bash setup.sh
```


### Historical setup / cleanup cell (archived, not executed)

The original environment-specific cell is preserved below. Configure the documented environment and workspace instead of running this cell.

```python
!pip install faiss-gpu-cu11==1.10.0
```


In [ ]:
import sys
sys.path.append('../data/workspace/multihop_dense_retrieval')

In [ ]:
from openai import OpenAI
import re
client = OpenAI(api_key=__import__("os").environ["OPENAI_API_KEY"])

def decompose_question_with_dependency(question: str, max_retries: int = 2):
    """
    복합 질문을 subquestion들로 분해하고, 의존성 및 paraphrase를 함께 반환합니다.
    비정상적인 dependency 구조일 경우 자동 재시도합니다.
    """
    def call_gpt(question):
        prompt = f"""
You are an expert at decomposing complex questions into subquestions with explicit, logically consistent placeholders and meaningful paraphrases.

──────────────── RULES ────────────────
1. Subquestion generation
   • Create as many subquestions as logically useful, up to a maximum of 3. Do not generate more than 3 subquestions under any circumstance.
   • For each subquestion, produce **one paraphrase** that keeps the same meaning but changes the wording.
   • Output format:
     Canonical – Paraphrase
   • Each sentence must be directly answerable and self-contained.
   • Never use vague pronouns such as “that book,” “those series,” etc.

2. Placeholder usage
   • When a subquestion requires the answer to a previous one, **always** refer to that exact answer using `[ANSWER_TO_SUBQUESTION_#]`.
   • You must refer only to the **immediately required answer**, not indirectly to earlier stages.
   • ✗ Do not say: “the companion books of [ANSWER_TO_SUBQUESTION_1]” if the companion books themselves were determined in subquestion 2.
   • ✓ Instead, directly use: “[ANSWER_TO_SUBQUESTION_2]”
   • This rule ensures that each subquestion has **minimal dependency scope** and enables clean step-by-step reasoning.

3. Question classification
   • Classify the question as either **bridge** or **comparison**.
     – **Bridge**: A step-by-step logical chain where later questions depend on earlier answers.
     – **Comparison**: Questions that require comparing two or more entities after gathering their individual properties.

4. Dependency tracking
   • Explicitly list which subquestion(s) depend on which earlier ones.

5. Dependency Code
   • After the Dependencies list, also print a compact dependency code string.
   • For each subquestion starting from 2, list the subquestion(s) it depends on:
     - Subquestion 2 depends on 1 → "1"
     - Subquestion 3 depends on 2 → "2"
     - Subquestion 3 depends on 1 and 2 → "12"
     - No dependency → "0"
   • Join them with hyphens: Example → "1-2" or "0-12"
   •✗ Never output codes like "0-01", "0-02", or "0-012" — 0 must not appear alongside any other digits in a dependency code. Each segment must reflect only immediate dependencies per subquestion.
   • ⚠ If the question is of type **comparison**, and the dependency code is "0-0", that likely means a final comparison step is missing. In such cases, strongly consider adding a third subquestion to explicitly compare the first two answers.
   • ⚠ If the dependency code is **"0-0"** and the question type is **bridge**, it may indicate that the question cannot be meaningfully decomposed. In such cases, consider that there might be **only one valid subquestion**.

──────────────── FEW-SHOT EXAMPLES ────────────────

❶ Bridge example
Question: “What book did the movie Dune originate from, and who illustrated the original cover?”
Question Type: bridge
Subquestions:
1. What is the title of the book that the movie Dune originated from? – Which book is the basis for the movie Dune?
2. Who illustrated the original cover of [ANSWER_TO_SUBQUESTION_1]? – Who was the cover illustrator of [ANSWER_TO_SUBQUESTION_1]?
Dependencies:
- Subquestion 2 depends on 1.
Dependency Code: 1

❸ Comparison example
Question: “Which country has better environmental policies: Sweden or Brazil?”
Question Type: comparison
Subquestions:
1. What are Sweden’s current environmental policies? – What kind of environmental regulations does Sweden have?
2. What are Brazil’s current environmental policies? – What environmental laws are in place in Brazil?
3. Which policies are more environmentally effective: [ANSWER_TO_SUBQUESTION_1] or [ANSWER_TO_SUBQUESTION_2]? – Which set of policies is greener: [ANSWER_TO_SUBQUESTION_1] or [ANSWER_TO_SUBQUESTION_2]?
Dependencies:
- Subquestion 3 depends on 1 and 2.
Dependency Code: 0-12

❹ Comparison example (complex)
Question: “Between Apple's iPhone and Samsung's Galaxy, which device had a larger market share in the year their 10th generation was released?”
Question Type: comparison
Subquestions:
1. What was the market share of Apple's iPhone in the year the 10th generation was released? – How much market share did Apple hold when the iPhone X came out?
2. What was the market share of Samsung’s Galaxy in the same year? – What was Samsung's market share during the release year of Galaxy S10?
3. Which had the larger market share: [ANSWER_TO_SUBQUESTION_1] or [ANSWER_TO_SUBQUESTION_2]? – Between [ANSWER_TO_SUBQUESTION_1] and [ANSWER_TO_SUBQUESTION_2], which was higher?
Dependencies:
- Subquestion 3 depends on 1 and 2.
Dependency Code: 0-12

──────────────── OUTPUT FORMAT ────────────────
Question Type: …
Subquestions:
1. Canonical – Paraphrase
2. Canonical – Paraphrase
3. Canonical – Paraphrase
Dependencies:
- …
Dependency Code: …

───────────────────────────────────────────────

Now decompose this question:

Question: "{question}"
"""

        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
        return response.choices[0].message.content

    def parse_output(output):
        lines = output.strip().split("\n")

        # 1. Question Type
        q_type = next((line.split(":")[1].strip() for line in lines if line.lower().startswith("question type")), None)

        # 2. Subquestions
        subq_lines = [line for line in lines if re.match(r'^\d+\.', line)]
        subqs = []
        for line in subq_lines:
            match = re.match(r'^(\d+)\.\s*(.*?)\s*–\s*(.*)', line)
            if match:
                idx, canonical, p1 = match.groups()
                subqs.append({
                    "index": int(idx),
                    "canonical": canonical.strip(),
                    "paraphrase": p1.strip()
                })

        # 3. Dependencies
        dependencies = [line[2:].strip() for line in lines if line.startswith("- ")]

        # 4. Dependency Code
        dep_code = None
        for line in lines:
            if line.lower().startswith("dependency code"):
                match = re.search(r':\s*(.*)', line)
                if match:
                    dep_code = match.group(1).strip()
                break

        return {
            "question": question,
            "type": q_type,
            "Subquestions": subqs,
            "dependencies": dependencies,
            "dependency_code": dep_code or ""
        }

    # 🔁 재시도 로직 (제한 없음)
    while True:
        output = call_gpt(question)
        print("🧠 GPT 응답 결과:\n", output)

        result = parse_output(output)

        if result is None:
            print("⚠️ GPT 응답 파싱 실패. 재시도합니다...\n")
            continue

        dep_code = result.get("dependency_code", "")
        q_type = result.get("type", "").lower()

        # ✅ 조건 검사: 문제 있는 구조면 재시도
        is_invalid_structure = (
            dep_code == "0-0"
        )

        if is_invalid_structure:
            print(f"⚠️ 재시도 필요 (Dependency Code: {dep_code}, Type: {q_type})\n")
            continue
        else:
            break

    return result



In [ ]:
import os
import json
import shutil

def save_decomposed_question(result: dict, save_root: str, question_id: str):
    """
    result: dict from decompose_question_with_dependency
    save_root: base path (e.g. "../data/workspace/testing")
    question_id: unique ID for this question (e.g. "q001")
    """

    # 1. 저장 경로 생성
    base_dir = os.path.join(save_root, question_id)
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)  # 기존 결과 삭제
    os.makedirs(base_dir)

    # 2. 복합질문 저장 (0/p.json)
    original_dir = os.path.join(base_dir, "0")
    os.makedirs(original_dir, exist_ok=True)
    original = {
        "_id": question_id,
        "question": result["question"],
        "type": result["type"],
        "subq_index": 0,
        "answer": ["Dummy Answer"],
        "sp": ["Dummy Source 1", "Dummy Source 2"],
        "dependencies": result["dependencies"],
        "dependency_code": result.get("dependency_code", None)
    }
    with open(os.path.join(original_dir, "p.json"), "w", encoding="utf-8") as f:
        json.dump(original, f, ensure_ascii=False, indent=2)

    # 3. 각 subQ + paraphrase 저장 (variant: 0=canonical, 1=paraphrase)
    for subq in result["Subquestions"]:
        subq_idx = subq["index"]

        for variant_idx in range(2):  # 0: canonical, 1: paraphrase
            variant_dir = os.path.join(base_dir, str(subq_idx), str(variant_idx))
            os.makedirs(variant_dir, exist_ok=True)

            question_text = subq["canonical"] if variant_idx == 0 else subq["paraphrase"]

            json_data = {
                "_id": question_id,
                "question": question_text,
                "subq_index": int(f"{subq_idx}{variant_idx}"),
                "answer": ["Dummy Answer"],
                "sp": ["Dummy Source 1", "Dummy Source 2"],
                "type": result["type"],
                "dependencies": result["dependencies"],
                "dependency_code": result.get("dependency_code", None)
            }

            with open(os.path.join(variant_dir, "p.json"), "w", encoding="utf-8") as f:
                json.dump(json_data, f, ensure_ascii=False, indent=2)

    print(f"✅ 저장 완료: {base_dir}")


In [ ]:
import json
import os
import shutil

def decompose_and_save_all(jsonl_path: str, save_root: str, start: int = 0, target_ids=None):
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    existing_ids = set(os.listdir(save_root))

    # target_ids가 문자열이면 리스트로 변환
    if target_ids is not None:
        if isinstance(target_ids, str):
            target_ids = [target_ids]
        target_ids = set(target_ids)

        # 🚩 target_ids가 있을 때: 해당 id만 찾아 처리하고 함수 종료!
        id_to_line = {}
        for idx, line in enumerate(lines):
            if not line.strip():
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                continue
            q_id = entry.get("_id")
            if q_id in target_ids:
                id_to_line[q_id] = (idx, entry)
            if len(id_to_line) == len(target_ids):
                break

        for q_id in target_ids:
            if q_id not in id_to_line:
                print(f"❌ {q_id}는 jsonl에 없음")
                continue
            idx, entry = id_to_line[q_id]
            q_text = entry["question"]

            save_path = os.path.join(save_root, q_id)
            if os.path.exists(save_path):
                if os.path.isdir(save_path):
                    shutil.rmtree(save_path)
                else:
                    os.remove(save_path)
                print(f"🗑️ 기존 결과({save_path}) 삭제 후 덮어쓰기 진행: {q_id}")

            print(f"\n🌀 Processing {q_id}... (index {idx})")
            try:
                result = decompose_question_with_dependency(q_text)
                save_decomposed_question(result, save_root, q_id)
                print(f"✅ 저장 완료 for {q_id}")
                print(f"Dependency code: {result.get('dependency_code', '❓없음')}")
            except Exception as e:
                print(f"❌ 오류 발생 (ID: {q_id}, index {idx}): {e}")
        return  # 🔥 반드시 함수 종료

    # ------ 이하 기존 전체 처리 루프 (target_ids=None일 때만) ------
    check_duplicates = True
    for idx, line in enumerate(lines):
        if not line.strip():
            continue
        try:
            entry = json.loads(line)
        except json.JSONDecodeError:
            print(f"⚠️ JSONDecodeError: index {idx} 줄은 JSON이 아닙니다.\n{line[:100]}")
            continue

        q_id = entry.get("_id")
        q_text = entry.get("question")

        if not q_id or not q_text:
            print(f"⚠️ index {idx}에 _id나 question 없음: {entry}")
            continue

        if idx < start:
            continue

        if check_duplicates and q_id in existing_ids:
            print(f"🚫 이미 존재 (ID: {q_id}), 스킵합니다. (index {idx})")
            continue
        else:
            if check_duplicates:
                print(f"🔄 중복 아닌 첫 케이스 발견 (ID: {q_id}, index {idx})\n    이후로는 중복 체크 없이 모두 처리합니다.")
                check_duplicates = False

        print(f"\n🌀 Processing {q_id}... (index {idx})")

        try:
            result = decompose_question_with_dependency(q_text)
            save_decomposed_question(result, save_root, q_id)
            print(f"✅ 저장 완료 for {q_id}")
            print(f"Dependency code: {result.get('dependency_code', '❓없음')}")
        except Exception as e:
            print(f"❌ 오류 발생 (ID: {q_id}, index {idx}): {e}")

In [ ]:
jsonl_path = "../data/workspace/hotpot_qas_val.json"
save_root = "../data/workspace/5000_hotpot_devide"

#decompose_and_save_all(jsonl_path, save_root,5322)
decompose_and_save_all(jsonl_path, save_root, target_ids="5a773c7755429972597f14b5")

In [ ]:
import os

base_path = "../data/workspace/5000_hotpot_devide"

# 폴더 개수만 카운트
num_dirs = sum(
    os.path.isdir(os.path.join(base_path, name))
    for name in os.listdir(base_path)
)

print(f"폴더 개수: {num_dirs}")

### Historical setup / cleanup cell (archived, not executed)

The original environment-specific cell is preserved below. Configure the documented environment and workspace instead of running this cell.

```python
import os
import shutil

base_path = "../data/workspace/5000_hotpot_devide"
folder_name = "5a86578755429960ec39b668"

folder_path = os.path.join(base_path, folder_name)

if os.path.isdir(folder_path):
    shutil.rmtree(folder_path)
    print(f"🗑️ 폴더 삭제 완료: {folder_path}")
else:
    print(f"❌ 폴더가 없습니다: {folder_path}")
```


In [ ]:
import os

base_path = "../data/workspace/5000_hotpot_devide"
folder_name = "5a86578755429960ec39b668"

folder_path = os.path.join(base_path, folder_name)

if os.path.isdir(folder_path):
    print(f"✅ 폴더가 존재합니다: {folder_path}")
else:
    print(f"❌ 폴더가 없습니다: {folder_path}")

In [ ]:
import os
import json

def generate_id_list(testing_root, output_path):
    """
    testing_root 하위의 폴더명을 기준으로 ID 리스트를 생성하여 저장
    """
    ids = [folder for folder in os.listdir(testing_root)
           if os.path.isdir(os.path.join(testing_root, folder))]
    ids.sort()  # 정렬하면 reproducibility 상승

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(ids, f, indent=2)

    print(f"✅ ID 리스트 저장 완료 ({len(ids)}개): {output_path}")


testing_root = "../data/workspace/testing"
output_path = "../data/workspace/question_ids.json"
generate_id_list(testing_root, output_path)

In [ ]:
import os
import json

def collect_questions_for_mdr_jsonl_single(input_path, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    with open(output_path, "w", encoding="utf-8") as out:
        out.write(json.dumps(data, ensure_ascii=False) + "\n")



In [ ]:
import subprocess

def run_step2_mdr_on_0json_all(testing_root: str, jsonl_output_root: str, mdr_output_root: str):
    """
    Step 2: 복합질문(0.json)만을 대상으로 MDR 실행
    """
    for qid in os.listdir(testing_root):
        base_dir = os.path.join(testing_root, qid)
        if not os.path.isdir(base_dir):
            continue

        print(f"\n🚀 Step 2 MDR 실행 대상 ID: {qid}")

        input_path = os.path.join(base_dir, "0.json")
        if not os.path.isfile(input_path):
            print(f"❌ 0.json 없음: {input_path}")
            continue

        # JSONL 경로
        jsonl_path = os.path.join(jsonl_output_root, f"{qid}_0.jsonl")
        output_path = os.path.join(mdr_output_root, f"{qid}_0.jsonl")

        # 🔧 출력 디렉토리 생성
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        os.makedirs(os.path.dirname(jsonl_path), exist_ok=True)

        # JSON → JSONL 변환
        with open(input_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        with open(jsonl_path, "w", encoding="utf-8") as out:
            out.write(json.dumps(data, ensure_ascii=False) + "\n")

        # MDR 실행
        cmd = f"""
        PYTHONPATH=../data/workspace/multihop_dense_retrieval \\
        python ../data/workspace/multihop_dense_retrieval/scripts/eval/eval_mhop_retrieval.py \\
            {jsonl_path} \\
            ../data/workspace/multihop_dense_retrieval/data/hotpot_index/wiki_index.npy \\
            ../data/workspace/multihop_dense_retrieval/data/hotpot_index/wiki_id2doc.json \\
            ../data/workspace/multihop_dense_retrieval/models/q_encoder.pt \\
            --batch-size 100 \\
            --beam-size 1 \\
            --topk 1 \\
            --shared-encoder \\
            --model-name roberta-base \\
            --save-path {output_path}
        """

        print(f"🔁 MDR 실행 중: {qid}")
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

        if result.returncode != 0:
            print(f"❌ MDR 실패 (ID: {qid})")
            print("📄 stdout:")
            print(result.stdout)
            print("⚠️ stderr:")
            print(result.stderr)
        else:
            print(f"✅ MDR 완료 (ID: {qid})")

In [ ]:
testing_root = "../data/workspace/testing"
jsonl_output_root = "../data/workspace/mdr_input"
mdr_output_root = "../data/workspace/mdr_results"

run_step2_mdr_on_0json_all(testing_root, jsonl_output_root, mdr_output_root)

기다루봐

위까지 일단 확정

In [ ]:
import json
import os
import re
import subprocess
from typing import Dict, List, Optional, Any
import logging

import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# 전역 변수로 모델 및 토크나이저 초기화 (메모리 효율성)
tokenizer = None
model = None
device = None

def initialize_qa_model():
    """QA 모델을 전역적으로 초기화"""
    global tokenizer, model, device

    if tokenizer is None or model is None:
        try:
            logger.info("Initializing UnifiedQA model...")
            tokenizer = T5Tokenizer.from_pretrained("allenai/unifiedqa-t5-large")
            model = T5ForConditionalGeneration.from_pretrained("allenai/unifiedqa-t5-large")
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model.to(device)
            logger.info(f"Model initialized on {device}")
        except Exception as e:
            logger.error(f"Failed to initialize QA model: {e}")
            raise e

def load_dependency_info(base_dir: str) -> Optional[str]:
    """0.json에서 dependency_code 읽기"""
    try:
        with open(os.path.join(base_dir, "0.json"), 'r', encoding='utf-8') as f:
            data = json.load(f)
            return data.get('dependency_code', '')
    except Exception as e:
        logger.error(f"Error loading dependency info from {base_dir}: {e}")
        return None

def parse_dependency_code(dep_code: str) -> Dict[int, List[int]]:
    """
    dependency_code 파싱
    예: "1-2" → {2: [1], 3: [2]}
    예: "0-12" → {2: [], 3: [1, 2]}
    예: "1" → {2: [1]}
    """
    if not dep_code:
        return {}

    dependencies = {}

    if '-' in dep_code:
        parts = dep_code.split('-')
        for i, part in enumerate(parts):
            subq_index = i + 2  # subQ2부터 시작
            if part == '0':
                dependencies[subq_index] = []
            else:
                # 각 숫자를 개별적으로 처리
                deps = [int(d) for d in part if d.isdigit()]
                dependencies[subq_index] = deps
    else:
        # 단일 숫자인 경우 (예: "1")
        deps = [int(d) for d in dep_code if d.isdigit()]
        dependencies[2] = deps  # subQ2가 subQ1에 의존

    return dependencies

def load_q_json(base_dir: str, subq_idx: int, variant: int) -> Optional[Dict]:
    """해당 subQ의 질문 데이터 로딩"""
    try:
        q_path = os.path.join(base_dir, str(subq_idx), str(variant), "q.json")
        with open(q_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        logger.error(f"Error loading question from {q_path}: {e}")
        return None

def replace_placeholders(text: str, answers_dict: Dict[int, str]) -> str:
    """[ANSWER_TO_SUBQUESTION_#] 문자열을 실제 값으로 치환"""
    if not text or not answers_dict:
        return text

    result = text
    pattern = r'\[ANSWER_TO_SUBQUESTION_(\d+)\]'

    def replace_match(match):
        subq_num = int(match.group(1))
        if subq_num in answers_dict:
            return answers_dict[subq_num]
        else:
            logger.warning(f"No answer found for SUBQUESTION_{subq_num}")
            return match.group(0)  # 원본 그대로 반환

    result = re.sub(pattern, replace_match, result)
    return result

def clear_gpu_memory():
    """GPU 메모리 정리"""
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            logger.info("GPU memory cleared")
    except Exception as e:
        logger.warning(f"Failed to clear GPU memory: {e}")

def run_mdr_on_qjson(q_json_path: str, save_path: str) -> bool:
    """
    q.json 파일에 대해 MDR 실행 (CUDA OOM 대응 포함)
    """
    try:
        # GPU 메모리 정리
        clear_gpu_memory()

        # 1. q.json -> input.jsonl 변환
        input_jsonl = q_json_path.replace("q.json", "input.jsonl")
        with open(q_json_path, 'r', encoding='utf-8') as f:
            q_data = json.load(f)

        with open(input_jsonl, 'w', encoding='utf-8') as f:
            f.write(json.dumps(q_data, ensure_ascii=False) + '\n')

        logger.info(f"Created input file: {input_jsonl}")

        # 2. MDR 명령어 실행 (메모리 최적화 옵션 포함)
        cmd = f"""
        CUDA_VISIBLE_DEVICES=0 \\
        PYTHONPATH=../data/workspace/multihop_dense_retrieval \\
        python ../data/workspace/multihop_dense_retrieval/scripts/eval/eval_mhop_retrieval.py \\
            {input_jsonl} \\
            ../data/workspace/multihop_dense_retrieval/data/hotpot_index/wiki_index.npy \\
            ../data/workspace/multihop_dense_retrieval/data/hotpot_index/wiki_id2doc.json \\
            ../data/workspace/multihop_dense_retrieval/models/q_encoder.pt \\
            --batch-size 16 --beam-size 1 --topk 1 --shared-encoder --model-name roberta-base \\
            --save-path {save_path}
        """

        logger.info(f"Running MDR command with reduced parameters")
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

        if result.returncode != 0:
            logger.warning(f"MDR failed with batch-size 16, trying CPU mode")
            # CPU 모드로 재시도
            return run_mdr_cpu_fallback(input_jsonl, save_path)

        # 3. 결과 파일 확인
        if os.path.exists(save_path):
            logger.info(f"MDR result saved to: {save_path}")
            return True
        else:
            logger.error(f"MDR result file not created: {save_path}")
            return False

    except Exception as e:
        logger.error(f"Exception in run_mdr_on_qjson: {e}")
        return False

def run_mdr_cpu_fallback(input_jsonl: str, save_path: str) -> bool:
    """
    CPU 모드로 MDR 실행 (CUDA OOM 발생 시 fallback)
    """
    try:
        logger.info("Attempting MDR execution on CPU")

        cmd = f"""
        CUDA_VISIBLE_DEVICES="" \\
        PYTHONPATH=../data/workspace/multihop_dense_retrieval \\
        python ../data/workspace/multihop_dense_retrieval/scripts/eval/eval_mhop_retrieval.py \\
            {input_jsonl} \\
            ../data/workspace/multihop_dense_retrieval/data/hotpot_index/wiki_index.npy \\
            ../data/workspace/multihop_dense_retrieval/data/hotpot_index/wiki_id2doc.json \\
            ../data/workspace/multihop_dense_retrieval/models/q_encoder.pt \\
            --batch-size 1 --beam-size 1 --topk 1 --shared-encoder --model-name roberta-base \\
            --save-path {save_path}
        """

        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

        if result.returncode != 0:
            logger.error(f"CPU MDR also failed with return code {result.returncode}")
            logger.error(f"STDERR: {result.stderr}")
            # 최후의 수단: 더미 결과 생성
            return create_dummy_mdr_result(save_path)

        if os.path.exists(save_path):
            logger.info(f"CPU MDR result saved to: {save_path}")
            return True
        else:
            logger.error(f"CPU MDR result file not created: {save_path}")
            return create_dummy_mdr_result(save_path)

    except Exception as e:
        logger.error(f"Exception in CPU fallback: {e}")
        return create_dummy_mdr_result(save_path)

def create_dummy_mdr_result(save_path: str) -> bool:
    """
    MDR이 완전히 실패한 경우 더미 결과 생성
    (QA가 context 없이도 동작하도록)
    """
    try:
        logger.warning(f"Creating dummy MDR result for {save_path}")

        dummy_result = {
            "candidate_chains": [[]],  # 빈 체인
            "passages": [],            # 빈 passage
            "scores": []
        }

        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(json.dumps(dummy_result, ensure_ascii=False) + '\n')

        logger.info(f"Dummy result created: {save_path}")
        return True

    except Exception as e:
        logger.error(f"Failed to create dummy result: {e}")
        return False

def load_mdr_result(path: str) -> str:
    """MDR 결과에서 context 추출 (실제 MDR 출력 형식에 맞춤)"""
    try:
        if not os.path.exists(path):
            logger.warning(f"MDR result file not found: {path}")
            return ""

        with open(path, 'r', encoding='utf-8') as f:
            line = f.readline().strip()
            if not line:
                logger.warning(f"Empty MDR result file: {path}")
                return ""

            data = json.loads(line)

            # MDR 결과 형식에 따라 context 추출
            # candidate_chains를 우선 시도
            chain = data.get("candidate_chains", [[]])[0]
            if chain:
                context = ' '.join([p.get("text", "") for p in chain])
                if context.strip():
                    return context.strip()

            # passages가 있는 경우
            passages = data.get('passages', [])
            if passages:
                context = ' '.join(passages[:3])  # 상위 3개 passage 사용
                if context.strip():
                    return context.strip()

            logger.warning(f"No valid context found in MDR result: {path}")
            return ""

    except json.JSONDecodeError as e:
        logger.error(f"JSON decode error in MDR result {path}: {e}")
        return ""
    except Exception as e:
        logger.error(f"Error loading MDR result from {path}: {e}")
        return ""

def qa_pipeline(question: str, context: str) -> Dict[str, str]:
    """
    UnifiedQA T5 모델을 사용하여 답변 생성
    """
    try:
        if not question.strip():
            return {"answer": "Error: Empty question"}

        # GPU 사용 가능 여부 확인
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # 토크나이저와 모델 로드 (전역 변수로 사용 권장)
        try:
            from transformers import T5Tokenizer, T5ForConditionalGeneration
            tokenizer = T5Tokenizer.from_pretrained("allenai/unifiedqa-t5-large")
            model = T5ForConditionalGeneration.from_pretrained("allenai/unifiedqa-t5-large")
            model.to(device)
        except Exception as model_error:
            logger.error(f"Model loading error: {model_error}")
            return {"answer": "Error: Model loading failed"}

        # 입력 텍스트 생성
        input_text = f"{question} \\n {context}" if context.strip() else question

        # 토큰화 및 길이 제한
        try:
            input_ids = tokenizer.encode(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
        except Exception as tokenize_error:
            logger.error(f"Tokenization error: {tokenize_error}")
            return {"answer": "Error: Tokenization failed"}

        # 답변 생성
        try:
            with torch.no_grad():
                outputs = model.generate(
                    input_ids,
                    max_length=32,
                    num_beams=4,
                    early_stopping=True,
                    pad_token_id=tokenizer.pad_token_id
                )
            answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

            if answer.strip():
                return {"answer": answer.strip()}
            else:
                return {"answer": "No answer generated"}

        except torch.cuda.OutOfMemoryError:
            logger.warning("CUDA OOM, falling back to CPU")
            model.to("cpu")
            input_ids = input_ids.to("cpu")
            with torch.no_grad():
                outputs = model.generate(
                    input_ids,
                    max_length=32,
                    num_beams=2,  # CPU에서는 beam 수 줄임
                    early_stopping=True
                )
            answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
            return {"answer": answer.strip() if answer.strip() else "No answer generated"}

    except Exception as e:
        logger.error(f"Error in QA pipeline: {e}")
        return {"answer": f"Error: {str(e)}"}

def check_mdr_dependencies():
    """MDR 실행에 필요한 파일들이 존재하는지 확인"""
    required_files = [
        "../data/workspace/multihop_dense_retrieval/scripts/eval/eval_mhop_retrieval.py",
        "../data/workspace/multihop_dense_retrieval/data/hotpot_index/wiki_index.npy",
        "../data/workspace/multihop_dense_retrieval/data/hotpot_index/wiki_id2doc.json",
        "../data/workspace/multihop_dense_retrieval/models/q_encoder.pt"
    ]

    missing_files = []
    for file_path in required_files:
        if not os.path.exists(file_path):
            missing_files.append(file_path)

    if missing_files:
        logger.error("MDR 실행에 필요한 파일들이 없습니다:")
        for file in missing_files:
            logger.error(f"  - {file}")
        return False

    logger.info("MDR 의존성 파일들이 모두 확인되었습니다.")
    return True

def run_step3_fused_qa_mdr_all_with_memory_management():
    """
    메모리 관리를 포함한 복합 질문 처리 함수
    """
    # 경로 설정
    base_path = "../data/workspace"
    testing_dir = os.path.join(base_path, "testing")
    mdr_results_dir = os.path.join(base_path, "mdr_results")
    qa_results_dir = os.path.join(base_path, "qa_results")
    question_ids_path = os.path.join(base_path, "question_ids.json")

    # 결과 저장을 위한 디렉토리 생성
    os.makedirs(mdr_results_dir, exist_ok=True)
    os.makedirs(qa_results_dir, exist_ok=True)

    # 첫 번째 UnifiedQA 모델을 먼저 언로드 (메모리 확보)
    global tokenizer, model, device
    if model is not None:
        logger.info("Unloading UnifiedQA model to free GPU memory for MDR")
        del model, tokenizer
        model, tokenizer = None, None
        clear_gpu_memory()

    # MDR 의존성 확인
    if not check_mdr_dependencies():
        logger.error("MDR 의존성 파일이 없어 실행을 중단합니다.")
        return

    # question_ids 로드
    try:
        with open(question_ids_path, 'r', encoding='utf-8') as f:
            question_ids = json.load(f)
        logger.info(f"Loaded {len(question_ids)} question IDs")
    except Exception as e:
        logger.error(f"Error loading question_ids.json: {e}")
        return

    all_answers = []
    failed_mdrs = []

    # Phase 1: MDR 실행만 먼저 처리
    logger.info("Phase 1: Running MDR for all questions")
    for i, question_id in enumerate(question_ids):
        logger.info(f"MDR Phase - Processing question_id: {question_id} ({i+1}/{len(question_ids)})")

        question_dir = os.path.join(testing_dir, question_id)
        if not os.path.exists(question_dir):
            continue

        dep_code = load_dependency_info(question_dir)
        if dep_code is None:
            continue

        dependencies = parse_dependency_code(dep_code)
        answers_dict = {}  # placeholder용

        for subq_index in range(1, 4):
            for variant in [0, 1]:
                subq_dir = os.path.join(question_dir, str(subq_index), str(variant))
                if not os.path.exists(subq_dir):
                    continue

                mdr_result_path = os.path.join(mdr_results_dir, f"{question_id}_{subq_index}{variant}.jsonl")

                if not os.path.exists(mdr_result_path):
                    q_json_path = os.path.join(subq_dir, "q.json")

                    # placeholder 치환 (필요한 경우)
                    if subq_index in dependencies and dependencies[subq_index]:
                        q_data = load_q_json(question_dir, subq_index, variant)
                        if q_data:
                            original_q = q_data.get('question', '')
                            processed_q = replace_placeholders(original_q, answers_dict)
                            if processed_q != original_q:
                                q_data["question"] = processed_q
                                with open(q_json_path, 'w', encoding='utf-8') as f:
                                    json.dump(q_data, f, ensure_ascii=False, indent=2)

                    logger.info(f"Running MDR for {question_id}_{subq_index}{variant}")
                    success = run_mdr_on_qjson(q_json_path, mdr_result_path)
                    if not success:
                        failed_mdrs.append(f"{question_id}_{subq_index}{variant}")

                    # 메모리 정리
                    clear_gpu_memory()

    # Phase 2: UnifiedQA 모델 로드 후 QA 실행
    logger.info("Phase 2: Loading UnifiedQA and running QA")
    initialize_qa_model()

    for i, question_id in enumerate(question_ids):
        logger.info(f"QA Phase - Processing question_id: {question_id} ({i+1}/{len(question_ids)})")

        question_dir = os.path.join(testing_dir, question_id)
        if not os.path.exists(question_dir):
            continue

        dep_code = load_dependency_info(question_dir)
        if dep_code is None:
            continue

        dependencies = parse_dependency_code(dep_code)
        answers_dict = {}

        for subq_index in range(1, 4):
            for variant in [0, 1]:
                subq_dir = os.path.join(question_dir, str(subq_index), str(variant))
                if not os.path.exists(subq_dir):
                    continue

                q_data = load_q_json(question_dir, subq_index, variant)
                if q_data is None:
                    continue

                original_question = q_data.get('question', '')
                if not original_question:
                    continue

                # placeholder 치환
                processed_question = original_question
                if subq_index in dependencies:
                    deps = dependencies[subq_index]
                    if deps:
                        processed_question = replace_placeholders(original_question, answers_dict)

                # MDR 결과 로드
                mdr_result_path = os.path.join(mdr_results_dir, f"{question_id}_{subq_index}{variant}.jsonl")
                context = load_mdr_result(mdr_result_path)

                # QA 실행
                qa_result = qa_pipeline(processed_question, context)
                answer = qa_result.get('answer', 'Unknown')

                # 답변 저장 (canonical만 재사용)
                if variant == 0:
                    answers_dict[subq_index] = answer

                # 결과 누적
                result_entry = {
                    "question_id": question_id,
                    "subq_index": subq_index,
                    "variant": variant,
                    "question": processed_question,
                    "original_question": original_question,
                    "answer": answer,
                    "has_context": bool(context.strip()),
                    "mdr_failed": f"{question_id}_{subq_index}{variant}" in failed_mdrs
                }
                all_answers.append(result_entry)

    # 결과 저장
    output_path = os.path.join(qa_results_dir, "subq_all_answers.json")
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(all_answers, f, indent=2, ensure_ascii=False)

        logger.info(f"✅ All answers saved to {output_path}")
        logger.info(f"📊 Statistics:")
        logger.info(f"  - Total processed entries: {len(all_answers)}")
        logger.info(f"  - Failed MDR runs: {len(failed_mdrs)}")
        logger.info(f"  - Success rate: {((len(all_answers) - len(failed_mdrs)) / len(all_answers) * 100):.1f}%")

        if failed_mdrs:
            logger.warning(f"⚠️ Failed MDRs: {failed_mdrs[:10]}...")  # 처음 10개만 표시

    except Exception as e:
        logger.error(f"Error saving results: {e}")

# 추가 헬퍼 함수
def retry_failed_mdrs():
    """실패한 MDR 작업들을 재시도하는 함수"""
    # 실패한 MDR 결과들을 다시 시도할 수 있는 유틸리티
    pass

if __name__ == "__main__":
    run_step3_fused_qa_mdr_all()